In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

print("Churn modeling started")

Churn modeling started


In [2]:
from pathlib import Path

CLEANED_DIR = Path("../data/cleaned")

customers = pd.read_csv(
    CLEANED_DIR / "customer_features.csv"
)

churn_labels = pd.read_csv(
    CLEANED_DIR / "churn_labels.csv"
)

print("Customer features shape:", customers.shape)
print("Churn labels shape:", churn_labels.shape)

Customer features shape: (8000, 35)
Churn labels shape: (8000, 4)


In [3]:
model_data = customers.merge(
    churn_labels[["customer_id", "churned"]],
    on="customer_id",
    how="left"
)

print("Modeling dataset shape:", model_data.shape)

print("\nTarget distribution:")
print(model_data["churned"].value_counts())

Modeling dataset shape: (8000, 36)

Target distribution:
churned
False    6009
True     1991
Name: count, dtype: int64


In [4]:
X = model_data.drop(
    columns=["customer_id", "churned"]
)

y = model_data["churned"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

Features shape: (8000, 34)
Target shape: (8000,)

Feature columns:
['first_name', 'last_name', 'age', 'gender', 'country', 'city', 'registration_date', 'acquisition_channel', 'customer_segment', 'preferred_language', 'tenure_days', 'total_watch_time_minutes', 'total_viewing_sessions', 'average_completion_percentage', 'average_rating', 'feedback_count', 'support_ticket_count', 'average_support_satisfaction', 'has_support_satisfaction', 'successful_payment_count', 'total_successful_payment_amount', 'failed_payment_count', 'subscription_count', 'has_plan_change', 'is_active_subscription', 'average_watch_time_per_session', 'unique_titles_watched', 'active_viewing_days', 'current_plan', 'payment_failure_rate', 'favorite_genre', 'unique_genres_watched', 'current_subscription_tenure_days', 'has_failed_payment']


In [5]:
print("Data types:")
print(X.dtypes)

print("\nMissing values:")
print(X.isnull().sum())

print("\nNumber of unique values:")
print(X.nunique())

Data types:
first_name                           object
last_name                            object
age                                 float64
gender                               object
country                              object
city                                 object
registration_date                    object
acquisition_channel                  object
customer_segment                     object
preferred_language                   object
tenure_days                           int64
total_watch_time_minutes            float64
total_viewing_sessions              float64
average_completion_percentage       float64
average_rating                      float64
feedback_count                      float64
support_ticket_count                float64
average_support_satisfaction        float64
has_support_satisfaction              int64
successful_payment_count            float64
total_successful_payment_amount     float64
failed_payment_count                float64
subscription_count  

In [6]:
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

numeric_columns = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

print("\nNumeric columns:")
print(numeric_columns)

print("\nColumns with missing values:")
print(X.isnull().sum()[X.isnull().sum() > 0])

Categorical columns:
['first_name', 'last_name', 'gender', 'country', 'city', 'registration_date', 'acquisition_channel', 'customer_segment', 'preferred_language', 'current_plan', 'favorite_genre']

Numeric columns:
['age', 'tenure_days', 'total_watch_time_minutes', 'total_viewing_sessions', 'average_completion_percentage', 'average_rating', 'feedback_count', 'support_ticket_count', 'average_support_satisfaction', 'has_support_satisfaction', 'successful_payment_count', 'total_successful_payment_amount', 'failed_payment_count', 'subscription_count', 'has_plan_change', 'is_active_subscription', 'average_watch_time_per_session', 'unique_titles_watched', 'active_viewing_days', 'payment_failure_rate', 'unique_genres_watched', 'current_subscription_tenure_days', 'has_failed_payment']

Columns with missing values:
average_support_satisfaction    4460
dtype: int64


In [7]:
X_model = X.drop(
    columns=[
        "first_name",
        "last_name",
        "registration_date"
    ]
)

print("Model features shape:", X_model.shape)

print("\nRemaining categorical columns:")
print(X_model.select_dtypes(include=["object"]).columns.tolist())

Model features shape: (8000, 31)

Remaining categorical columns:
['gender', 'country', 'city', 'acquisition_channel', 'customer_segment', 'preferred_language', 'current_plan', 'favorite_genre']


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining churn distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True))

X_train: (6400, 31)
X_test: (1600, 31)
y_train: (6400,)
y_test: (1600,)

Training churn distribution:
churned
False    0.751094
True     0.248906
Name: proportion, dtype: float64

Testing churn distribution:
churned
False    0.75125
True     0.24875
Name: proportion, dtype: float64


In [9]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:", len(categorical_features))

print("\nNumber of numeric features:", len(numeric_features))

Categorical features:
['gender', 'country', 'city', 'acquisition_channel', 'customer_segment', 'preferred_language', 'current_plan', 'favorite_genre']

Number of categorical features: 8

Number of numeric features: 23


In [10]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

numeric_imputer = SimpleImputer(
    strategy="median"
)

print("Preprocessing components created.")

Preprocessing components created.


In [11]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_imputer,
            numeric_features
        ),
        (
            "categorical",
            categorical_encoder,
            categorical_features
        )
    ]
)

print("Preprocessor created.")

Preprocessor created.


In [12]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (6400, 243)
Processed testing shape: (1600, 243)


In [13]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_processed,
    y_train
)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


c:\Users\91830\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_pred = logistic_model.predict(X_test_processed)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0

Confusion Matrix:
[[1202    0]
 [   0  398]]

Classification Report:
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      1202
        True       1.00      1.00      1.00       398

    accuracy                           1.00      1600
   macro avg       1.00      1.00      1.00      1600
weighted avg       1.00      1.00      1.00      1600



In [15]:
leakage_check = model_data.groupby("churned")[
    [
        "has_plan_change",
        "is_active_subscription",
        "subscription_count",
        "current_subscription_tenure_days"
    ]
].mean()

print(leakage_check)

         has_plan_change  is_active_subscription  subscription_count  \
churned                                                                
False           0.105841                     1.0            1.105841   
True            0.120040                     0.0            1.120040   

         current_subscription_tenure_days  
churned                                    
False                         1329.270261  
True                          1202.099950  


In [16]:
X_safe = X_model.drop(
    columns=[
        "is_active_subscription",
        "has_plan_change"
    ]
)

print("Original features:", X_model.shape[1])
print("Safer features:", X_safe.shape[1])

print("\nRemoved features:")
print(["is_active_subscription", "has_plan_change"])

Original features: 31
Safer features: 29

Removed features:
['is_active_subscription', 'has_plan_change']


In [17]:
X_train_safe, X_test_safe, y_train_safe, y_test_safe = train_test_split(
    X_safe,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train_safe:", X_train_safe.shape)
print("X_test_safe:", X_test_safe.shape)
print("y_train_safe:", y_train_safe.shape)
print("y_test_safe:", y_test_safe.shape)

X_train_safe: (6400, 29)
X_test_safe: (1600, 29)
y_train_safe: (6400,)
y_test_safe: (1600,)


In [18]:
categorical_features_safe = X_train_safe.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_features_safe = X_train_safe.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical features:", categorical_features_safe)
print("\nNumber of categorical features:", len(categorical_features_safe))

print("\nNumber of numeric features:", len(numeric_features_safe))

print("\nMissing numeric values:")
print(
    X_train_safe[numeric_features_safe]
    .isnull()
    .sum()[lambda x: x > 0]
)

Categorical features: ['gender', 'country', 'city', 'acquisition_channel', 'customer_segment', 'preferred_language', 'current_plan', 'favorite_genre']

Number of categorical features: 8

Number of numeric features: 21

Missing numeric values:
average_support_satisfaction    3569
dtype: int64


In [19]:
categorical_encoder_safe = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

numeric_imputer_safe = SimpleImputer(
    strategy="median"
)

preprocessor_safe = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_imputer_safe,
            numeric_features_safe
        ),
        (
            "categorical",
            categorical_encoder_safe,
            categorical_features_safe
        )
    ]
)

print("Safer preprocessor created.")

Safer preprocessor created.


In [20]:
X_train_safe_processed = preprocessor_safe.fit_transform(
    X_train_safe
)

X_test_safe_processed = preprocessor_safe.transform(
    X_test_safe
)

print("Processed training shape:", X_train_safe_processed.shape)
print("Processed testing shape:", X_test_safe_processed.shape)

Processed training shape: (6400, 241)
Processed testing shape: (1600, 241)


In [21]:
logistic_model_safe = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model_safe.fit(
    X_train_safe_processed,
    y_train_safe
)

print("Safer Logistic Regression model trained successfully.")

Safer Logistic Regression model trained successfully.


c:\Users\91830\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
logistic_model_safe = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model_safe.fit(
    X_train_safe_processed,
    y_train_safe
)

print("Safer Logistic Regression model trained successfully.")

Safer Logistic Regression model trained successfully.


c:\Users\91830\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
y_pred_safe = logistic_model_safe.predict(
    X_test_safe_processed
)

print("Accuracy:", accuracy_score(y_test_safe, y_pred_safe))
print("Precision:", precision_score(y_test_safe, y_pred_safe))
print("Recall:", recall_score(y_test_safe, y_pred_safe))
print("F1-score:", f1_score(y_test_safe, y_pred_safe))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_safe, y_pred_safe))

print("\nClassification Report:")
print(
    classification_report(
        y_test_safe,
        y_pred_safe
    )
)

Accuracy: 0.75125
Precision: 0.5
Recall: 0.02512562814070352
F1-score: 0.04784688995215311

Confusion Matrix:
[[1192   10]
 [ 388   10]]

Classification Report:
              precision    recall  f1-score   support

       False       0.75      0.99      0.86      1202
        True       0.50      0.03      0.05       398

    accuracy                           0.75      1600
   macro avg       0.63      0.51      0.45      1600
weighted avg       0.69      0.75      0.66      1600



In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features_safe
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features_safe
        )
    ]
)

print("Scaled preprocessor created.")

Scaled preprocessor created.


In [25]:
X_train_scaled = preprocessor_scaled.fit_transform(
    X_train_safe
)

X_test_scaled = preprocessor_scaled.transform(
    X_test_safe
)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

Scaled training shape: (6400, 241)
Scaled testing shape: (1600, 241)


In [26]:
logistic_model_scaled = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logistic_model_scaled.fit(
    X_train_scaled,
    y_train_safe
)

print("Scaled Logistic Regression model trained successfully.")

Scaled Logistic Regression model trained successfully.


In [27]:
y_pred_scaled = logistic_model_scaled.predict(
    X_test_scaled
)

print("Accuracy:", accuracy_score(y_test_safe, y_pred_scaled))
print("Precision:", precision_score(y_test_safe, y_pred_scaled))
print("Recall:", recall_score(y_test_safe, y_pred_scaled))
print("F1-score:", f1_score(y_test_safe, y_pred_scaled))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_safe, y_pred_scaled))

print("\nClassification Report:")
print(
    classification_report(
        y_test_safe,
        y_pred_scaled
    )
)


Accuracy: 0.745
Precision: 0.41935483870967744
Recall: 0.06532663316582915
F1-score: 0.11304347826086956

Confusion Matrix:
[[1166   36]
 [ 372   26]]

Classification Report:
              precision    recall  f1-score   support

       False       0.76      0.97      0.85      1202
        True       0.42      0.07      0.11       398

    accuracy                           0.74      1600
   macro avg       0.59      0.52      0.48      1600
weighted avg       0.67      0.74      0.67      1600



In [28]:
logistic_model_balanced = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logistic_model_balanced.fit(
    X_train_scaled,
    y_train_safe
)

print("Balanced Logistic Regression model trained successfully.")

Balanced Logistic Regression model trained successfully.


In [29]:
y_pred_balanced = logistic_model_balanced.predict(
    X_test_scaled
)

print("Accuracy:", accuracy_score(y_test_safe, y_pred_balanced))
print("Precision:", precision_score(y_test_safe, y_pred_balanced))
print("Recall:", recall_score(y_test_safe, y_pred_balanced))
print("F1-score:", f1_score(y_test_safe, y_pred_balanced))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_safe, y_pred_balanced))

print("\nClassification Report:")
print(
    classification_report(
        y_test_safe,
        y_pred_balanced
    )
)

Accuracy: 0.59
Precision: 0.3125
Recall: 0.5402010050251256
F1-score: 0.39594843462246776

Confusion Matrix:
[[729 473]
 [183 215]]

Classification Report:
              precision    recall  f1-score   support

       False       0.80      0.61      0.69      1202
        True       0.31      0.54      0.40       398

    accuracy                           0.59      1600
   macro avg       0.56      0.57      0.54      1600
weighted avg       0.68      0.59      0.62      1600



In [30]:
prediction_cutoff = pd.Timestamp("2026-06-30")

print("Prediction cutoff:", prediction_cutoff)

Prediction cutoff: 2026-06-30 00:00:00


In [31]:
model_data = model_data.drop(
    columns=["churn_date", "observation_date"],
    errors="ignore"
)

churn_dates = churn_labels[
    ["customer_id", "churn_date"]
].copy()

churn_dates["churn_date"] = pd.to_datetime(
    churn_dates["churn_date"]
)

model_data = model_data.merge(
    churn_dates,
    on="customer_id",
    how="left"
)

model_data["observation_date"] = (
    model_data["churn_date"]
    .fillna(prediction_cutoff)
)

print(
    model_data[
        ["customer_id", "churned", "churn_date", "observation_date"]
    ].head(10)
)

print("\nObservation dates missing:")

print(model_data["observation_date"].isnull().sum())

  customer_id  churned churn_date observation_date
0  CUST000744    False        NaT       2026-06-30
1  CUST001502    False        NaT       2026-06-30
2  CUST005383    False        NaT       2026-06-30
3  CUST004708     True 2026-01-08       2026-01-08
4  CUST005877    False        NaT       2026-06-30
5  CUST003762    False        NaT       2026-06-30
6  CUST007892    False        NaT       2026-06-30
7  CUST002317    False        NaT       2026-06-30
8  CUST005609    False        NaT       2026-06-30
9  CUST000519    False        NaT       2026-06-30

Observation dates missing:
0


In [32]:
viewing_activity = pd.read_csv(
    CLEANED_DIR / "viewing_activity.csv"
)

payments = pd.read_csv(
    CLEANED_DIR / "payments.csv"
)

support_tickets = pd.read_csv(
    CLEANED_DIR / "support_tickets.csv"
)

customer_feedback = pd.read_csv(
    CLEANED_DIR / "customer_feedback.csv"
)

subscriptions = pd.read_csv(
    CLEANED_DIR / "subscriptions.csv"
)

print("Viewing activity:", viewing_activity.shape)
print("Payments:", payments.shape)
print("Support tickets:", support_tickets.shape)
print("Customer feedback:", customer_feedback.shape)
print("Subscriptions:", subscriptions.shape)

Viewing activity: (66422, 9)
Payments: (110735, 8)
Support tickets: (6120, 11)
Customer feedback: (5046, 6)
Subscriptions: (8875, 10)


In [33]:
viewing_activity["viewing_date"] = pd.to_datetime(
    viewing_activity["viewing_date"]
)

payments["payment_date"] = pd.to_datetime(
    payments["payment_date"]
)

support_tickets["ticket_date"] = pd.to_datetime(
    support_tickets["ticket_date"]
)

customer_feedback["feedback_date"] = pd.to_datetime(
    customer_feedback["feedback_date"]
)

subscriptions["subscription_start_date"] = pd.to_datetime(
    subscriptions["subscription_start_date"]
)

subscriptions["subscription_end_date"] = pd.to_datetime(
    subscriptions["subscription_end_date"]
)

subscriptions["cancellation_date"] = pd.to_datetime(
    subscriptions["cancellation_date"]
)

print("Event dates converted to datetime.")

Event dates converted to datetime.


In [34]:
viewing_point_in_time = viewing_activity.merge(
    model_data[
        ["customer_id", "observation_date"]
    ],
    on="customer_id",
    how="inner"
)

viewing_point_in_time = viewing_point_in_time[
    viewing_point_in_time["viewing_date"]
    <= viewing_point_in_time["observation_date"]
].copy()

print(
    "Viewing records before/at observation date:",
    viewing_point_in_time.shape
)

print(
    "\nLatest viewing date:",
    viewing_point_in_time["viewing_date"].max()
)

Viewing records before/at observation date: (60859, 10)

Latest viewing date: 2026-06-30 00:00:00


In [35]:
viewing_features_pt = (
    viewing_point_in_time
    .groupby("customer_id")
    .agg(
        total_watch_time_minutes=("watch_duration_minutes", "sum"),
        total_viewing_sessions=("viewing_id", "count"),
        average_completion_percentage=("completion_percentage", "mean"),
        unique_titles_watched=("content_id", "nunique"),
        active_viewing_days=("viewing_date", "nunique")
    )
    .reset_index()
)

print("Point-in-time viewing features:")
print(viewing_features_pt.head())

print("\nShape:", viewing_features_pt.shape)

Point-in-time viewing features:
  customer_id  total_watch_time_minutes  total_viewing_sessions  \
0  CUST000001                       226                       5   
1  CUST000002                       206                       5   
2  CUST000003                        83                       2   
3  CUST000004                       469                       8   
4  CUST000005                       397                      10   

   average_completion_percentage  unique_titles_watched  active_viewing_days  
0                          77.20                      5                    5  
1                          65.60                      5                    5  
2                          76.50                      2                    2  
3                          63.75                      8                    8  
4                          70.00                     10                    9  

Shape: (7848, 6)


In [36]:
payments_point_in_time = payments.merge(
    model_data[
        ["customer_id", "observation_date"]
    ],
    on="customer_id",
    how="inner"
)

payments_point_in_time = payments_point_in_time[
    payments_point_in_time["payment_date"]
    <= payments_point_in_time["observation_date"]
].copy()

print(
    "Payment records before/at observation date:",
    payments_point_in_time.shape
)

print(
    "\nLatest payment date:",
    payments_point_in_time["payment_date"].max()
)

Payment records before/at observation date: (109630, 9)

Latest payment date: 2026-05-31 00:00:00


In [37]:
payment_features_pt = (
    payments_point_in_time
    .groupby("customer_id")
    .agg(
        successful_payment_count=("payment_status", lambda x: (x == "Success").sum()),
        total_successful_payment_amount=(
            "amount",
            lambda x: x[payments_point_in_time.loc[x.index, "payment_status"] == "Success"].sum()
        ),
        failed_payment_count=("payment_status", lambda x: (x == "Failed").sum())
    )
    .reset_index()
)

payment_features_pt["payment_failure_rate"] = (
    payment_features_pt["failed_payment_count"]
    /
    (
        payment_features_pt["successful_payment_count"]
        + payment_features_pt["failed_payment_count"]
    )
)

payment_features_pt["payment_failure_rate"] = (
    payment_features_pt["payment_failure_rate"]
    .fillna(0)
)

payment_features_pt["has_failed_payment"] = (
    payment_features_pt["failed_payment_count"] > 0
).astype(int)

print(payment_features_pt.head())

print("\nShape:", payment_features_pt.shape)


  customer_id  successful_payment_count  total_successful_payment_amount  \
0  CUST000001                        15                           333.35   
1  CUST000002                         6                            92.94   
2  CUST000003                        15                           344.85   
3  CUST000004                        14                            97.86   
4  CUST000005                        14                            97.86   

   failed_payment_count  payment_failure_rate  has_failed_payment  
0                     0              0.000000                   0  
1                     0              0.000000                   0  
2                     0              0.000000                   0  
3                     1              0.066667                   1  
4                     1              0.066667                   1  

Shape: (8000, 6)


In [38]:
support_point_in_time = support_tickets.merge(
    model_data[
        ["customer_id", "observation_date"]
    ],
    on="customer_id",
    how="inner"
)

support_point_in_time = support_point_in_time[
    support_point_in_time["ticket_date"]
    <= support_point_in_time["observation_date"]
].copy()

print(
    "Support tickets before/at observation date:",
    support_point_in_time.shape
)

print(
    "\nLatest ticket date:",
    support_point_in_time["ticket_date"].max()
)

Support tickets before/at observation date: (5767, 12)

Latest ticket date: 2026-06-30 00:00:00


In [39]:
support_features_pt = (
    support_point_in_time
    .groupby("customer_id")
    .agg(
        support_ticket_count=("ticket_id", "count"),
        average_support_satisfaction=(
            "customer_satisfaction_score",
            "mean"
        )
    )
    .reset_index()
)

support_features_pt["has_support_satisfaction"] = (
    support_features_pt["average_support_satisfaction"]
    .notna()
).astype(int)

print(support_features_pt.head())

print("\nShape:", support_features_pt.shape)

print("\nMissing satisfaction scores:")
print(
    support_features_pt[
        "average_support_satisfaction"
    ].isnull().sum()
)

  customer_id  support_ticket_count  average_support_satisfaction  \
0  CUST000001                     2                           2.5   
1  CUST000002                     1                           3.0   
2  CUST000003                     1                           NaN   
3  CUST000007                     1                           3.0   
4  CUST000010                     1                           5.0   

   has_support_satisfaction  
0                         1  
1                         1  
2                         0  
3                         1  
4                         1  

Shape: (4229, 4)

Missing satisfaction scores:
836


In [40]:
feedback_point_in_time = customer_feedback.merge(
    model_data[
        ["customer_id", "observation_date"]
    ],
    on="customer_id",
    how="inner"
)

feedback_point_in_time = feedback_point_in_time[
    feedback_point_in_time["feedback_date"]
    <= feedback_point_in_time["observation_date"]
].copy()

print(
    "Feedback records before/at observation date:",
    feedback_point_in_time.shape
)

print(
    "\nLatest feedback date:",
    feedback_point_in_time["feedback_date"].max()
)

Feedback records before/at observation date: (4827, 7)

Latest feedback date: 2026-06-30 00:00:00


In [41]:
feedback_features_pt = (
    feedback_point_in_time
    .groupby("customer_id")
    .agg(
        average_rating=("rating", "mean"),
        feedback_count=("feedback_id", "count")
    )
    .reset_index()
)

print(feedback_features_pt.head())

print("\nShape:", feedback_features_pt.shape)

print("\nCustomers with feedback:")
print(feedback_features_pt["customer_id"].nunique())

  customer_id  average_rating  feedback_count
0  CUST000001             3.0               2
1  CUST000002             1.0               1
2  CUST000003             3.0               1
3  CUST000008             3.0               1
4  CUST000009             4.0               1

Shape: (3440, 3)

Customers with feedback:
3440


In [42]:
subscriptions_point_in_time = subscriptions.merge(
    model_data[
        ["customer_id", "observation_date"]
    ],
    on="customer_id",
    how="inner"
)

subscriptions_point_in_time = subscriptions_point_in_time[
    subscriptions_point_in_time["subscription_start_date"]
    <= subscriptions_point_in_time["observation_date"]
].copy()

print(
    "Subscription records before/at observation date:",
    subscriptions_point_in_time.shape
)

print(
    "\nLatest subscription start date:",
    subscriptions_point_in_time[
        "subscription_start_date"
    ].max()
)

Subscription records before/at observation date: (8833, 11)

Latest subscription start date: 2026-06-29 00:00:00


In [43]:
subscription_features_pt = (
    subscriptions_point_in_time
    .groupby("customer_id")
    .agg(
        subscription_count=("subscription_id", "count")
    )
    .reset_index()
)

# Get the most recent subscription record available
latest_subscription_pt = (
    subscriptions_point_in_time
    .sort_values(
        ["customer_id", "subscription_start_date"]
    )
    .drop_duplicates(
        "customer_id",
        keep="last"
    )
)

latest_subscription_pt = latest_subscription_pt[
    [
        "customer_id",
        "plan_id",
        "subscription_start_date"
    ]
].copy()

# Add the latest plan to the aggregated features
subscription_features_pt = subscription_features_pt.merge(
    latest_subscription_pt,
    on="customer_id",
    how="left"
)

# Calculate tenure of the latest subscription as of observation date
subscription_features_pt["current_subscription_tenure_days"] = (
    subscription_features_pt["customer_id"]
    .map(
        model_data.set_index("customer_id")["observation_date"]
    )
    - subscription_features_pt["subscription_start_date"]
).dt.days

print(subscription_features_pt.head())

print("\nShape:", subscription_features_pt.shape)

  customer_id  subscription_count plan_id subscription_start_date  \
0  CUST000001                   1  PLAN03              2022-04-30   
1  CUST000002                   1  PLAN02              2025-12-24   
2  CUST000003                   1  PLAN03              2019-12-15   
3  CUST000004                   1  PLAN01              2019-01-28   
4  CUST000005                   1  PLAN01              2024-07-24   

   current_subscription_tenure_days  
0                              1522  
1                               188  
2                              2318  
3                              2710  
4                               706  

Shape: (8000, 5)


In [44]:
point_in_time_data = model_data[
    [
        "customer_id",
        "churned",
        "observation_date"
    ]
].copy()

point_in_time_data = point_in_time_data.merge(
    viewing_features_pt,
    on="customer_id",
    how="left"
)

point_in_time_data = point_in_time_data.merge(
    payment_features_pt,
    on="customer_id",
    how="left"
)

point_in_time_data = point_in_time_data.merge(
    support_features_pt,
    on="customer_id",
    how="left"
)

point_in_time_data = point_in_time_data.merge(
    feedback_features_pt,
    on="customer_id",
    how="left"
)

point_in_time_data = point_in_time_data.merge(
    subscription_features_pt[
        [
            "customer_id",
            "subscription_count",
            "plan_id",
            "current_subscription_tenure_days"
        ]
    ],
    on="customer_id",
    how="left"
)

print("Point-in-time dataset shape:", point_in_time_data.shape)

print("\nMissing values:")
print(
    point_in_time_data.isnull().sum()[
        point_in_time_data.isnull().sum() > 0
    ]
)

Point-in-time dataset shape: (8000, 21)

Missing values:
total_watch_time_minutes          152
total_viewing_sessions            152
average_completion_percentage     152
unique_titles_watched             152
active_viewing_days               152
support_ticket_count             3771
average_support_satisfaction     4607
has_support_satisfaction         3771
average_rating                   4560
feedback_count                   4560
dtype: int64


In [45]:
print("Viewing missing:")
print(
    point_in_time_data[
        [
            "total_watch_time_minutes",
            "total_viewing_sessions",
            "average_completion_percentage",
            "unique_titles_watched",
            "active_viewing_days"
        ]
    ].isnull().sum()
)

print("\nPayment missing:")
print(
    point_in_time_data[
        [
            "successful_payment_count",
            "total_successful_payment_amount",
            "failed_payment_count",
            "payment_failure_rate",
            "has_failed_payment"
        ]
    ].isnull().sum()
)

print("\nSupport missing:")
print(
    point_in_time_data[
        [
            "support_ticket_count",
            "average_support_satisfaction",
            "has_support_satisfaction"
        ]
    ].isnull().sum()
)

print("\nFeedback missing:")
print(
    point_in_time_data[
        [
            "average_rating",
            "feedback_count"
        ]
    ].isnull().sum()
)

print("\nSubscription missing:")
print(
    point_in_time_data[
        [
            "subscription_count",
            "plan_id",
            "current_subscription_tenure_days"
        ]
    ].isnull().sum()
)

Viewing missing:
total_watch_time_minutes         152
total_viewing_sessions           152
average_completion_percentage    152
unique_titles_watched            152
active_viewing_days              152
dtype: int64

Payment missing:
successful_payment_count           0
total_successful_payment_amount    0
failed_payment_count               0
payment_failure_rate               0
has_failed_payment                 0
dtype: int64

Support missing:
support_ticket_count            3771
average_support_satisfaction    4607
has_support_satisfaction        3771
dtype: int64

Feedback missing:
average_rating    4560
feedback_count    4560
dtype: int64

Subscription missing:
subscription_count                  0
plan_id                             0
current_subscription_tenure_days    0
dtype: int64


In [46]:
# Viewing: no activity means zero activity
viewing_columns = [
    "total_watch_time_minutes",
    "total_viewing_sessions",
    "average_completion_percentage",
    "unique_titles_watched",
    "active_viewing_days"
]

point_in_time_data[viewing_columns] = (
    point_in_time_data[viewing_columns].fillna(0)
)

# Support: no ticket means zero tickets
point_in_time_data["support_ticket_count"] = (
    point_in_time_data["support_ticket_count"].fillna(0)
)

point_in_time_data["has_support_satisfaction"] = (
    point_in_time_data["has_support_satisfaction"].fillna(0)
)

# Feedback: no feedback means zero feedback records
point_in_time_data["feedback_count"] = (
    point_in_time_data["feedback_count"].fillna(0)
)

print("Structural missing values handled.")

print("\nRemaining missing values:")
print(
    point_in_time_data.isnull().sum()[
        point_in_time_data.isnull().sum() > 0
    ]
)

Structural missing values handled.

Remaining missing values:
average_support_satisfaction    4607
average_rating                  4560
dtype: int64


In [47]:
X_pt = point_in_time_data.drop(
    columns=[
        "customer_id",
        "churned",
        "observation_date"
    ]
)

y_pt = point_in_time_data["churned"]

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_pt,
    y_pt,
    test_size=0.20,
    random_state=42,
    stratify=y_pt
)

print("X_train_pt:", X_train_pt.shape)
print("X_test_pt:", X_test_pt.shape)
print("y_train_pt:", y_train_pt.shape)
print("y_test_pt:", y_test_pt.shape)

print("\nTraining churn distribution:")
print(y_train_pt.value_counts(normalize=True))

print("\nTesting churn distribution:")
print(y_test_pt.value_counts(normalize=True))

X_train_pt: (6400, 18)
X_test_pt: (1600, 18)
y_train_pt: (6400,)
y_test_pt: (1600,)

Training churn distribution:
churned
False    0.751094
True     0.248906
Name: proportion, dtype: float64

Testing churn distribution:
churned
False    0.75125
True     0.24875
Name: proportion, dtype: float64


In [48]:
churn_date_summary = churn_labels.copy()

churn_date_summary["churn_date"] = pd.to_datetime(
    churn_date_summary["churn_date"]
)

print("Total customers:", len(churn_date_summary))

print(
    "\nChurned customers:",
    churn_date_summary["churned"].sum()
)

print(
    "\nChurn dates range:",
    churn_date_summary["churn_date"].min(),
    "to",
    churn_date_summary["churn_date"].max()
)

print("\nChurns by month:")
print(
    churn_date_summary[
        churn_date_summary["churned"] == True
    ]
    .set_index("churn_date")
    .resample("ME")
    .size()
)

Total customers: 8000

Churned customers: 1991

Churn dates range: 2025-10-03 00:00:00 to 2026-06-30 00:00:00

Churns by month:
churn_date
2025-10-31    198
2025-11-30    200
2025-12-31    213
2026-01-31    253
2026-02-28    190
2026-03-31    213
2026-04-30    223
2026-05-31    250
2026-06-30    251
Freq: ME, dtype: int64


In [49]:
prediction_cutoff = pd.Timestamp("2026-03-31")
prediction_end = pd.Timestamp("2026-06-30")

print("Prediction cutoff:", prediction_cutoff)
print("Prediction window:", prediction_cutoff + pd.Timedelta(days=1), "to", prediction_end)

# Customers who churned during the future prediction window
future_churns = churn_labels[
    (churn_labels["churned"] == True) &
    (pd.to_datetime(churn_labels["churn_date"]) > prediction_cutoff) &
    (pd.to_datetime(churn_labels["churn_date"]) <= prediction_end)
]

print("\nFuture churners:", len(future_churns))

Prediction cutoff: 2026-03-31 00:00:00
Prediction window: 2026-04-01 00:00:00 to 2026-06-30 00:00:00

Future churners: 724


In [50]:
churn_dates = pd.to_datetime(
    churn_labels["churn_date"]
)

eligible_customers = churn_labels[
    (churn_labels["churned"] == False) |
    (churn_dates > prediction_cutoff)
].copy()

eligible_customers["future_churn"] = (
    (eligible_customers["churned"] == True) &
    (churn_dates > prediction_cutoff) &
    (churn_dates <= prediction_end)
).astype(int)

print("Eligible customers:", len(eligible_customers))

print("\nFuture churners:", eligible_customers["future_churn"].sum())

print("\nFuture churn distribution:")
print(
    eligible_customers["future_churn"].value_counts()
)

print("\nFuture churn rate:")
print(
    eligible_customers["future_churn"].mean()
)

Eligible customers: 6733

Future churners: 724

Future churn distribution:
future_churn
0    6009
1     724
Name: count, dtype: int64

Future churn rate:
0.10753007574632407


In [51]:
model_data_pt = eligible_customers[
    ["customer_id", "future_churn"]
].copy()

model_data_pt["observation_date"] = prediction_cutoff

print("Modeling dataset shape:", model_data_pt.shape)

print("\nTarget distribution:")
print(model_data_pt["future_churn"].value_counts())

print("\nTarget rate:")
print(model_data_pt["future_churn"].mean())

Modeling dataset shape: (6733, 3)

Target distribution:
future_churn
0    6009
1     724
Name: count, dtype: int64

Target rate:
0.10753007574632407


In [52]:
customer_base = customers[
    [
        "customer_id",
        "gender",
        "country",
        "city",
        "acquisition_channel",
        "customer_segment",
        "preferred_language",
        "registration_date"
    ]
].copy()

customer_base["registration_date"] = pd.to_datetime(
    customer_base["registration_date"]
)

customer_base["tenure_days"] = (
    prediction_cutoff - customer_base["registration_date"]
).dt.days

print("Customer base shape:", customer_base.shape)

print("\nTenure statistics:")
print(customer_base["tenure_days"].describe())

Customer base shape: (8000, 9)

Tenure statistics:
count    8000.000000
mean     1284.986250
std       776.164124
min       -61.000000
25%       615.750000
50%      1285.000000
75%      1949.000000
max      2646.000000
Name: tenure_days, dtype: float64


In [53]:
future_registered = customer_base[
    customer_base["tenure_days"] < 0
]

print("Customers registered after prediction cutoff:",
      len(future_registered))

print("\nRegistration date range:")
print(future_registered["registration_date"].min())
print(future_registered["registration_date"].max())

print("\nFuture-registered customers in eligible cohort:")
print(
    future_registered["customer_id"]
    .isin(model_data_pt["customer_id"])
    .sum()
)

Customers registered after prediction cutoff: 162

Registration date range:
2026-04-01 00:00:00
2026-05-31 00:00:00

Future-registered customers in eligible cohort:
162


In [54]:
eligible_ids = model_data_pt[
    model_data_pt["customer_id"].isin(
        customer_base.loc[
            customer_base["tenure_days"] >= 0,
            "customer_id"
        ]
    )
].copy()

print("Eligible customers after registration-date check:",
      len(eligible_ids))

print("\nFuture churners:")
print(eligible_ids["future_churn"].sum())

print("\nFuture churn rate:")
print(eligible_ids["future_churn"].mean())

Eligible customers after registration-date check: 6571

Future churners:
684

Future churn rate:
0.10409374524425506


In [55]:
viewing = pd.read_csv(
    Path("../data/cleaned/viewing_activity.csv")
)

viewing["viewing_date"] = pd.to_datetime(
    viewing["viewing_date"]
)

print("Viewing data shape:", viewing.shape)
print("Date range:", viewing["viewing_date"].min(), "to", viewing["viewing_date"].max())

Viewing data shape: (66422, 9)
Date range: 2025-07-01 00:00:00 to 2026-06-30 00:00:00


In [56]:
viewing_pt = viewing[
    viewing["viewing_date"] <= prediction_cutoff
].copy()

print("Viewing records available by cutoff:", len(viewing_pt))

print("\nDate range:")
print(viewing_pt["viewing_date"].min())
print(viewing_pt["viewing_date"].max())

print("\nUnique customers:")
print(viewing_pt["customer_id"].nunique())

Viewing records available by cutoff: 44349

Date range:
2025-07-01 00:00:00
2026-03-31 00:00:00

Unique customers:
7588


In [57]:
viewing_features = viewing_pt.groupby("customer_id").agg(
    total_watch_time_minutes=("watch_duration_minutes", "sum"),
    total_viewing_sessions=("viewing_id", "count"),
    average_completion_percentage=("completion_percentage", "mean"),
    unique_titles_watched=("content_id", "nunique"),
    active_viewing_days=("viewing_date", "nunique")
).reset_index()

print("Viewing features shape:", viewing_features.shape)

print("\nFirst 5 rows:")
print(viewing_features.head())

Viewing features shape: (7588, 6)

First 5 rows:
  customer_id  total_watch_time_minutes  total_viewing_sessions  \
0  CUST000001                       106                       3   
1  CUST000002                        90                       2   
2  CUST000003                        83                       2   
3  CUST000004                       251                       5   
4  CUST000005                       295                       7   

   average_completion_percentage  unique_titles_watched  active_viewing_days  
0                      83.000000                      3                    3  
1                      63.000000                      2                    2  
2                      76.500000                      2                    2  
3                      59.000000                      5                    5  
4                      69.714286                      7                    6  


In [58]:
model_data_pt = model_data_pt.merge(
    viewing_features,
    on="customer_id",
    how="left"
)

print("Modeling dataset shape:", model_data_pt.shape)

print("\nMissing viewing features:")
print(
    model_data_pt[
        [
            "total_watch_time_minutes",
            "total_viewing_sessions",
            "average_completion_percentage",
            "unique_titles_watched",
            "active_viewing_days"
        ]
    ].isna().sum()
)

Modeling dataset shape: (6733, 8)

Missing viewing features:
total_watch_time_minutes         364
total_viewing_sessions           364
average_completion_percentage    364
unique_titles_watched            364
active_viewing_days              364
dtype: int64


In [59]:
viewing_columns = [
    "total_watch_time_minutes",
    "total_viewing_sessions",
    "average_completion_percentage",
    "unique_titles_watched",
    "active_viewing_days"
]

model_data_pt[viewing_columns] = model_data_pt[viewing_columns].fillna(0)

print("Missing viewing features after filling:")
print(model_data_pt[viewing_columns].isna().sum())

print("\nModeling dataset shape:")
print(model_data_pt.shape)

Missing viewing features after filling:
total_watch_time_minutes         0
total_viewing_sessions           0
average_completion_percentage    0
unique_titles_watched            0
active_viewing_days              0
dtype: int64

Modeling dataset shape:
(6733, 8)


In [60]:
model_data_pt = eligible_ids.copy()

print("Corrected modeling dataset shape:", model_data_pt.shape)

print("\nTarget distribution:")
print(model_data_pt["future_churn"].value_counts())

print("\nFuture churn rate:")
print(model_data_pt["future_churn"].mean())

Corrected modeling dataset shape: (6571, 3)

Target distribution:
future_churn
0    5887
1     684
Name: count, dtype: int64

Future churn rate:
0.10409374524425506


In [61]:
model_data_pt = model_data_pt.merge(
    viewing_features,
    on="customer_id",
    how="left"
)

print("Modeling dataset shape:", model_data_pt.shape)

print("\nMissing viewing features:")
print(
    model_data_pt[viewing_columns].isna().sum()
)


Modeling dataset shape: (6571, 8)

Missing viewing features:
total_watch_time_minutes         202
total_viewing_sessions           202
average_completion_percentage    202
unique_titles_watched            202
active_viewing_days              202
dtype: int64


In [62]:
model_data_pt[viewing_columns] = model_data_pt[viewing_columns].fillna(0)

print("Missing viewing features after filling:")
print(model_data_pt[viewing_columns].isna().sum())

print("\nFinal shape after viewing features:")
print(model_data_pt.shape)

Missing viewing features after filling:
total_watch_time_minutes         0
total_viewing_sessions           0
average_completion_percentage    0
unique_titles_watched            0
active_viewing_days              0
dtype: int64

Final shape after viewing features:
(6571, 8)


In [63]:
payments = pd.read_csv(
    Path("../data/cleaned/payments.csv")
)

payments["payment_date"] = pd.to_datetime(
    payments["payment_date"]
)

print("Payments data shape:", payments.shape)

print(
    "Date range:",
    payments["payment_date"].min(),
    "to",
    payments["payment_date"].max()
)

Payments data shape: (110735, 8)
Date range: 2019-01-01 00:00:00 to 2026-05-31 00:00:00


In [64]:
payments_pt = payments[
    payments["payment_date"] <= prediction_cutoff
].copy()

print("Payment records available by cutoff:", len(payments_pt))

print("\nDate range:")
print(
    payments_pt["payment_date"].min(),
    "to",
    payments_pt["payment_date"].max()
)

print("\nUnique customers:")
print(payments_pt["customer_id"].nunique())

Payment records available by cutoff: 108043

Date range:
2019-01-01 00:00:00 to 2026-03-31 00:00:00

Unique customers:
7838


In [65]:
payment_features = payments_pt.groupby("customer_id").agg(
    successful_payment_count=(
        "payment_status",
        lambda x: (x == "Success").sum()
    ),
    total_successful_payment_amount=(
        "amount",
        lambda x: x[payments_pt.loc[x.index, "payment_status"] == "Success"].sum()
    ),
    failed_payment_count=(
        "payment_status",
        lambda x: (x == "Failed").sum()
    ),
    total_payment_count=(
        "payment_id",
        "count"
    )
).reset_index()

payment_features["payment_failure_rate"] = (
    payment_features["failed_payment_count"]
    / payment_features["total_payment_count"]
)

payment_features["has_failed_payment"] = (
    payment_features["failed_payment_count"] > 0
).astype(int)

print("Payment features shape:", payment_features.shape)

print("\nFirst 5 rows:")
print(payment_features.head())

Payment features shape: (7838, 7)

First 5 rows:
  customer_id  successful_payment_count  total_successful_payment_amount  \
0  CUST000001                        15                           333.35   
1  CUST000002                         4                            61.96   
2  CUST000003                        15                           344.85   
3  CUST000004                        14                            97.86   
4  CUST000005                        14                            97.86   

   failed_payment_count  total_payment_count  payment_failure_rate  \
0                     0                   15              0.000000   
1                     0                    4              0.000000   
2                     0                   15              0.000000   
3                     1                   15              0.066667   
4                     1                   15              0.066667   

   has_failed_payment  
0                   0  
1                   0  
2

In [66]:
model_data_pt = model_data_pt.merge(
    payment_features,
    on="customer_id",
    how="left"
)

payment_columns = [
    "successful_payment_count",
    "total_successful_payment_amount",
    "failed_payment_count",
    "total_payment_count",
    "payment_failure_rate",
    "has_failed_payment"
]

print("Modeling dataset shape:", model_data_pt.shape)

print("\nMissing payment features:")
print(model_data_pt[payment_columns].isna().sum())

Modeling dataset shape: (6571, 14)

Missing payment features:
successful_payment_count           0
total_successful_payment_amount    0
failed_payment_count               0
total_payment_count                0
payment_failure_rate               0
has_failed_payment                 0
dtype: int64


In [67]:
support = pd.read_csv(
    Path("../data/cleaned/support_tickets.csv")
)

support["ticket_date"] = pd.to_datetime(
    support["ticket_date"]
)

print("Support data shape:", support.shape)

print(
    "Date range:",
    support["ticket_date"].min(),
    "to",
    support["ticket_date"].max()
)

Support data shape: (6120, 11)
Date range: 2019-03-03 00:00:00 to 2026-06-30 00:00:00


In [68]:
support_pt = support[
    support["ticket_date"] <= prediction_cutoff
].copy()

print("Support tickets available by cutoff:", len(support_pt))

print("\nDate range:")
print(
    support_pt["ticket_date"].min(),
    "to",
    support_pt["ticket_date"].max()
)

print("\nUnique customers:")
print(support_pt["customer_id"].nunique())

Support tickets available by cutoff: 5206

Date range:
2019-03-03 00:00:00 to 2026-03-31 00:00:00

Unique customers:
3862


In [69]:
support_features = support_pt.groupby("customer_id").agg(
    support_ticket_count=(
        "ticket_id",
        "count"
    ),
    average_support_satisfaction=(
        "customer_satisfaction_score",
        "mean"
    )
).reset_index()

support_features["has_support_satisfaction"] = (
    support_features["average_support_satisfaction"].notna()
).astype(int)

print("Support features shape:", support_features.shape)

print("\nFirst 5 rows:")
print(support_features.head())

Support features shape: (3862, 4)

First 5 rows:
  customer_id  support_ticket_count  average_support_satisfaction  \
0  CUST000001                     2                           2.5   
1  CUST000003                     1                           NaN   
2  CUST000007                     1                           3.0   
3  CUST000010                     1                           5.0   
4  CUST000011                     1                           3.0   

   has_support_satisfaction  
0                         1  
1                         0  
2                         1  
3                         1  
4                         1  


In [70]:
model_data_pt = model_data_pt.merge(
    support_features,
    on="customer_id",
    how="left"
)

print("Modeling dataset shape:", model_data_pt.shape)

support_columns = [
    "support_ticket_count",
    "average_support_satisfaction",
    "has_support_satisfaction"
]

print("\nMissing support features:")
print(model_data_pt[support_columns].isna().sum())

Modeling dataset shape: (6571, 17)

Missing support features:
support_ticket_count            3322
average_support_satisfaction    3980
has_support_satisfaction        3322
dtype: int64


In [71]:
model_data_pt["support_ticket_count"] = (
    model_data_pt["support_ticket_count"].fillna(0)
)

model_data_pt["has_support_satisfaction"] = (
    model_data_pt["has_support_satisfaction"].fillna(0)
)

print("Missing support ticket count:",
      model_data_pt["support_ticket_count"].isna().sum())

print("Missing satisfaction indicator:",
      model_data_pt["has_support_satisfaction"].isna().sum())

print("Missing satisfaction score:",
      model_data_pt["average_support_satisfaction"].isna().sum())

Missing support ticket count: 0
Missing satisfaction indicator: 0
Missing satisfaction score: 3980


In [72]:
feedback = pd.read_csv(
    Path("../data/cleaned/customer_feedback.csv")
)

feedback["feedback_date"] = pd.to_datetime(
    feedback["feedback_date"]
)

print("Feedback data shape:", feedback.shape)

print(
    "Date range:",
    feedback["feedback_date"].min(),
    "to",
    feedback["feedback_date"].max()
)

Feedback data shape: (5046, 6)
Date range: 2019-03-06 00:00:00 to 2026-06-30 00:00:00


In [73]:
feedback_pt = feedback[
    feedback["feedback_date"] <= prediction_cutoff
].copy()

print("Feedback records available by cutoff:", len(feedback_pt))

print("\nDate range:")
print(
    feedback_pt["feedback_date"].min(),
    "to",
    feedback_pt["feedback_date"].max()
)

print("\nUnique customers:")
print(feedback_pt["customer_id"].nunique())

Feedback records available by cutoff: 4374

Date range:
2019-03-06 00:00:00 to 2026-03-31 00:00:00

Unique customers:
3192


In [75]:
feedback_features = feedback_pt.groupby("customer_id").agg(
    average_rating=("rating", "mean"),
    feedback_count=("feedback_id", "count")
).reset_index()

print("Feedback features shape:", feedback_features.shape)

print("\nFirst 5 rows:")
print(feedback_features.head())

Feedback features shape: (3192, 3)

First 5 rows:
  customer_id  average_rating  feedback_count
0  CUST000001             3.0               2
1  CUST000003             3.0               1
2  CUST000008             3.0               1
3  CUST000009             4.0               1
4  CUST000013             4.0               1


In [76]:
model_data_pt = model_data_pt.merge(
    feedback_features,
    on="customer_id",
    how="left"
)

feedback_columns = [
    "average_rating",
    "feedback_count"
]

print("Modeling dataset shape:", model_data_pt.shape)

print("\nMissing feedback features:")
print(model_data_pt[feedback_columns].isna().sum())

Modeling dataset shape: (6571, 19)

Missing feedback features:
average_rating    3908
feedback_count    3908
dtype: int64


In [77]:
model_data_pt["feedback_count"] = (
    model_data_pt["feedback_count"].fillna(0)
)

print("Missing feedback count:",
      model_data_pt["feedback_count"].isna().sum())

print("Missing average rating:",
      model_data_pt["average_rating"].isna().sum())

Missing feedback count: 0
Missing average rating: 3908


In [78]:
subscriptions = pd.read_csv(
    Path("../data/cleaned/subscriptions.csv")
)

subscriptions["subscription_start_date"] = pd.to_datetime(
    subscriptions["subscription_start_date"]
)

subscriptions["subscription_end_date"] = pd.to_datetime(
    subscriptions["subscription_end_date"]
)

subscriptions["cancellation_date"] = pd.to_datetime(
    subscriptions["cancellation_date"]
)

print("Subscriptions data shape:", subscriptions.shape)

print(
    "Subscription start range:",
    subscriptions["subscription_start_date"].min(),
    "to",
    subscriptions["subscription_start_date"].max()
)

Subscriptions data shape: (8875, 10)
Subscription start range: 2019-01-01 00:00:00 to 2026-06-29 00:00:00


In [79]:
subscriptions_pt = subscriptions[
    subscriptions["subscription_start_date"] <= prediction_cutoff
].copy()

print("Subscription records available by cutoff:", len(subscriptions_pt))
print("Unique customers:", subscriptions_pt["customer_id"].nunique())
print(
    "Date range:",
    subscriptions_pt["subscription_start_date"].min(),
    "to",
    subscriptions_pt["subscription_start_date"].max()
)

Subscription records available by cutoff: 8624
Unique customers: 7838
Date range: 2019-01-01 00:00:00 to 2026-03-31 00:00:00


In [80]:
subscription_features = subscriptions_pt.groupby("customer_id").agg(
    subscription_count=("subscription_id", "count"),
    latest_subscription_start=("subscription_start_date", "max")
).reset_index()

subscription_features["current_subscription_tenure_days"] = (
    prediction_cutoff -
    subscription_features["latest_subscription_start"]
).dt.days

print("Subscription features shape:", subscription_features.shape)
print(subscription_features.head())

Subscription features shape: (7838, 4)
  customer_id  subscription_count latest_subscription_start  \
0  CUST000001                   1                2022-04-30   
1  CUST000002                   1                2025-12-24   
2  CUST000003                   1                2019-12-15   
3  CUST000004                   1                2019-01-28   
4  CUST000005                   1                2024-07-24   

   current_subscription_tenure_days  
0                              1431  
1                                97  
2                              2298  
3                              2619  
4                               615  


In [81]:
latest_plan = (
    subscriptions_pt
    .sort_values(["customer_id", "subscription_start_date"])
    .groupby("customer_id")
    .tail(1)
    [["customer_id", "plan_id"]]
)

subscription_features = subscription_features.merge(
    latest_plan,
    on="customer_id",
    how="left"
)

print(subscription_features.head())

  customer_id  subscription_count latest_subscription_start  \
0  CUST000001                   1                2022-04-30   
1  CUST000002                   1                2025-12-24   
2  CUST000003                   1                2019-12-15   
3  CUST000004                   1                2019-01-28   
4  CUST000005                   1                2024-07-24   

   current_subscription_tenure_days plan_id  
0                              1431  PLAN03  
1                                97  PLAN02  
2                              2298  PLAN03  
3                              2619  PLAN01  
4                               615  PLAN01  


In [82]:
model_data_pt = model_data_pt.merge(
    subscription_features,
    on="customer_id",
    how="left"
)

print("Modeling dataset shape:", model_data_pt.shape)
print("\nMissing subscription features:")
print(
    model_data_pt[
        [
            "subscription_count",
            "current_subscription_tenure_days",
            "plan_id"
        ]
    ].isna().sum()
)

Modeling dataset shape: (6571, 23)

Missing subscription features:
subscription_count                  0
current_subscription_tenure_days    0
plan_id                             0
dtype: int64


In [83]:
model_data_pt = model_data_pt.merge(
    customer_base[
        [
            "customer_id",
            "gender",
            "country",
            "city",
            "acquisition_channel",
            "customer_segment",
            "preferred_language",
            "tenure_days"
        ]
    ],
    on="customer_id",
    how="left"
)

print("Final feature dataset shape:", model_data_pt.shape)

print("\nColumns:")
print(model_data_pt.columns.tolist())

Final feature dataset shape: (6571, 30)

Columns:
['customer_id', 'future_churn', 'observation_date', 'total_watch_time_minutes', 'total_viewing_sessions', 'average_completion_percentage', 'unique_titles_watched', 'active_viewing_days', 'successful_payment_count', 'total_successful_payment_amount', 'failed_payment_count', 'total_payment_count', 'payment_failure_rate', 'has_failed_payment', 'support_ticket_count', 'average_support_satisfaction', 'has_support_satisfaction', 'average_rating', 'feedback_count', 'subscription_count', 'latest_subscription_start', 'current_subscription_tenure_days', 'plan_id', 'gender', 'country', 'city', 'acquisition_channel', 'customer_segment', 'preferred_language', 'tenure_days']


In [84]:
leakage_columns = [
    "churned",
    "churn_date",
    "churn_reason",
    "cancellation_date",
    "cancellation_reason",
    "status",
    "is_active_subscription",
    "has_plan_change"
]

print("Potential leakage columns present:")
print(
    [col for col in leakage_columns if col in model_data_pt.columns]
)

print("\nDuplicate customer IDs:",
      model_data_pt["customer_id"].duplicated().sum())

print("\nMissing values:")
print(model_data_pt.isna().sum().sort_values(ascending=False).head(10))

Potential leakage columns present:
[]

Duplicate customer IDs: 0

Missing values:
average_support_satisfaction       3980
average_rating                     3908
observation_date                      0
total_watch_time_minutes              0
customer_id                           0
future_churn                          0
unique_titles_watched                 0
active_viewing_days                   0
successful_payment_count              0
total_successful_payment_amount       0
dtype: int64


In [85]:
X = model_data_pt.drop(
    columns=["customer_id", "future_churn", "observation_date"]
)

y = model_data_pt["future_churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nCategorical columns:")
print(X.select_dtypes(include="object").columns.tolist())

print("\nNumeric columns:")
print(X.select_dtypes(exclude="object").columns.tolist())

X shape: (6571, 27)
y shape: (6571,)

Target distribution:
future_churn
0    5887
1     684
Name: count, dtype: int64

Categorical columns:
['plan_id', 'gender', 'country', 'city', 'acquisition_channel', 'customer_segment', 'preferred_language']

Numeric columns:
['total_watch_time_minutes', 'total_viewing_sessions', 'average_completion_percentage', 'unique_titles_watched', 'active_viewing_days', 'successful_payment_count', 'total_successful_payment_amount', 'failed_payment_count', 'total_payment_count', 'payment_failure_rate', 'has_failed_payment', 'support_ticket_count', 'average_support_satisfaction', 'has_support_satisfaction', 'average_rating', 'feedback_count', 'subscription_count', 'latest_subscription_start', 'current_subscription_tenure_days', 'tenure_days']


In [86]:
X = X.drop(columns=["latest_subscription_start"])

print("X shape after removing raw date:", X.shape)

print("\nRemaining datetime columns:")
print(
    X.select_dtypes(include=["datetime64[ns]"]).columns.tolist()
)

X shape after removing raw date: (6571, 26)

Remaining datetime columns:
[]


In [87]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTraining churn rate:", y_train.mean())
print("Testing churn rate:", y_test.mean())

X_train: (5256, 26)
X_test: (1315, 26)

Training churn rate: 0.10407153729071537
Testing churn rate: 0.10418250950570342


In [88]:
categorical_columns = X_train.select_dtypes(
    include="object"
).columns.tolist()

numeric_columns = X_train.select_dtypes(
    exclude="object"
).columns.tolist()

print("Categorical features:", len(categorical_columns))
print(categorical_columns)

print("\nNumeric features:", len(numeric_columns))
print(numeric_columns)

Categorical features: 7
['plan_id', 'gender', 'country', 'city', 'acquisition_channel', 'customer_segment', 'preferred_language']

Numeric features: 19
['total_watch_time_minutes', 'total_viewing_sessions', 'average_completion_percentage', 'unique_titles_watched', 'active_viewing_days', 'successful_payment_count', 'total_successful_payment_amount', 'failed_payment_count', 'total_payment_count', 'payment_failure_rate', 'has_failed_payment', 'support_ticket_count', 'average_support_satisfaction', 'has_support_satisfaction', 'average_rating', 'feedback_count', 'subscription_count', 'current_subscription_tenure_days', 'tenure_days']


In [89]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [90]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (5256, 222)
Processed testing shape: (1315, 222)


In [91]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_processed,
    y_train
)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [92]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

y_pred = logistic_model.predict(X_test_processed)
y_prob = logistic_model.predict_proba(X_test_processed)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.6060836501901141
Precision: 0.10721649484536082
Recall   : 0.3795620437956204
F1 Score : 0.16720257234726688
ROC-AUC  : 0.5433061108150645

Confusion Matrix:
[[745 433]
 [ 85  52]]


In [93]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

print("Random Forest training completed.")

Random Forest training completed.


In [94]:
rf_pred = random_forest_model.predict(X_test_processed)
rf_prob = random_forest_model.predict_proba(X_test_processed)[:, 1]

print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))
print("ROC-AUC  :", roc_auc_score(y_test, rf_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

Accuracy : 0.8859315589353612
Precision: 0.24
Recall   : 0.043795620437956206
F1 Score : 0.07407407407407407
ROC-AUC  : 0.580821756534024

Confusion Matrix:
[[1159   19]
 [ 131    6]]


In [95]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("Threshold | Precision | Recall | F1")

for threshold in thresholds:
    threshold_pred = (rf_prob >= threshold).astype(int)

    print(
        f"{threshold:9.2f} | "
        f"{precision_score(y_test, threshold_pred, zero_division=0):9.3f} | "
        f"{recall_score(y_test, threshold_pred):6.3f} | "
        f"{f1_score(y_test, threshold_pred):5.3f}"
    )

Threshold | Precision | Recall | F1
     0.10 |     0.106 |  0.883 | 0.190
     0.15 |     0.118 |  0.737 | 0.204
     0.20 |     0.128 |  0.547 | 0.208
     0.25 |     0.130 |  0.358 | 0.190
     0.30 |     0.157 |  0.255 | 0.194
     0.35 |     0.188 |  0.182 | 0.185
     0.40 |     0.193 |  0.117 | 0.145
     0.45 |     0.220 |  0.080 | 0.118
     0.50 |     0.269 |  0.051 | 0.086


In [96]:
final_threshold = 0.20

final_pred = (rf_prob >= final_threshold).astype(int)

print("Final threshold:", final_threshold)
print("Accuracy :", accuracy_score(y_test, final_pred))
print("Precision:", precision_score(y_test, final_pred))
print("Recall   :", recall_score(y_test, final_pred))
print("F1 Score :", f1_score(y_test, final_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_pred))

Final threshold: 0.2
Accuracy : 0.5657794676806084
Precision: 0.1284246575342466
Recall   : 0.5474452554744526
F1 Score : 0.20804438280166435

Confusion Matrix:
[[669 509]
 [ 62  75]]


In [97]:
import joblib
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

joblib.dump(
    random_forest_model,
    models_dir / "churn_random_forest.pkl"
)

joblib.dump(
    preprocessor,
    models_dir / "churn_preprocessor.pkl"
)

print("Model saved:")
print(models_dir / "churn_random_forest.pkl")

print("\nPreprocessor saved:")
print(models_dir / "churn_preprocessor.pkl")

Model saved:
..\models\churn_random_forest.pkl

Preprocessor saved:
..\models\churn_preprocessor.pkl


In [99]:
# Transform all eligible customers using the fitted preprocessor
X_all_processed = preprocessor.transform(X)

# Generate churn probabilities for all customers
all_probabilities = random_forest_model.predict_proba(
    X_all_processed
)[:, 1]

# Create customer risk table
risk_scores = model_data_pt[
    ["customer_id", "future_churn"]
].copy()

risk_scores["churn_probability"] = all_probabilities

risk_scores["risk_level"] = pd.cut(
    risk_scores["churn_probability"],
    bins=[-1, 0.20, 0.50, 1.00],
    labels=["Low", "Medium", "High"]
)

print("Risk score table shape:", risk_scores.shape)

print("\nRisk distribution:")
print(risk_scores["risk_level"].value_counts())

print("\nTop 10 highest-risk customers:")
print(
    risk_scores
    .sort_values("churn_probability", ascending=False)
    .head(10)
)

Risk score table shape: (6571, 4)

Risk distribution:
risk_level
Low       4994
Medium    1005
High       572
Name: count, dtype: int64

Top 10 highest-risk customers:
     customer_id  future_churn  churn_probability risk_level
3415  CUST004181             1                1.0       High
434   CUST000543             1                1.0       High
3097  CUST003790             1                1.0       High
1676  CUST002057             1                1.0       High
1718  CUST002117             1                1.0       High
4945  CUST006013             1                1.0       High
4958  CUST006030             1                1.0       High
6221  CUST007574             1                1.0       High
5953  CUST007245             1                1.0       High
501   CUST000622             1                1.0       High


In [100]:
risk_output = risk_scores[
    ["customer_id", "churn_probability", "risk_level"]
].copy()

risk_output = risk_output.sort_values(
    "churn_probability",
    ascending=False
)

risk_output.to_csv(
    "../data/cleaned/customer_churn_risk.csv",
    index=False
)

print("Risk table saved successfully.")
print("Rows:", len(risk_output))

print("\nSaved to:")
print("../data/cleaned/customer_churn_risk.csv")

Risk table saved successfully.
Rows: 6571

Saved to:
../data/cleaned/customer_churn_risk.csv
